# Module 4 — Data Flow in Applications

## We will progressively build one backend system.

# Step 1 — Create CSV File Manually

Create a file named: `items.csv`
Add the following content manually:

In [2]:
print("""id,name,price,quantity
1,Laptop,50000,2
2,Phone,20000,5
3,Mouse,500,10""")

id,name,price,quantity
1,Laptop,50000,2
2,Phone,20000,5
3,Mouse,500,10


# Step 2 — Install Dependencies

In [ ]:
print("Run this in terminal:")
print("pip install fastapi uvicorn pandas")

# Step 3 — Create Initial Backend

In [ ]:
from fastapi import FastAPI
import pandas as pd

app = FastAPI()

@app.get("/items")
def get_items():

    # Read CSV file
    df = pd.read_csv("items.csv")

    # Convert dataframe to JSON-compatible format
    data = df.to_dict(orient="records")

    return data

In [ ]:
import pandas as pd
pd.read_csv('items.csv')    

,id,name,price,quantity
0,1,Laptop,50000,2
1,2,Phone,20000,5
2,3,Mouse,500,10


# Step 4 — Run Backend

Save this inside: `main.py`

Run: `uvicorn main:app --reload`

# Step 5 — Test API

Open: `http://127.0.0.1:8000/items`

Observe: CSV data is returned as JSON.

# Step 6 — Understanding Persistence

CSV storage persists after restart.
Variables in RAM do not.

In [ ]:
counter = 0

@app.get("/counter")
def counter_api():
    global counter
    counter += 1
    return {
        "counter": counter
    }

# Add the above code BELOW the existing code.

Test: `http://127.0.0.1:8000/counter`

Refresh multiple times. Then restart server.

Observe: Counter resets.

# Step 7 — CREATE Operation

Now we will add POST API.
This will add new rows into CSV.

In [ ]:
from pydantic import BaseModel

# Add this import BELOW existing imports.

In [ ]:
class Item(BaseModel):
    name: str
    price: float
    quantity: int

# Add this BELOW: `app = FastAPI()`

In [ ]:
@app.post("/items")
def create_item(item: Item):

    # Read existing CSV
    df = pd.read_csv("items.csv")

    # Generate new ID
    if len(df) > 0:
        new_id = df["id"].max() + 1
    else:
        new_id = 1

    # Create new row
    new_item = {
        "id": new_id,
        "name": item.name,
        "price": item.price,
        "quantity": item.quantity
    }

    # Append row
    df.loc[len(df)] = new_item

    # Save updated CSV
    df.to_csv("items.csv", index=False)

    return {
        "message": "Item added successfully",
        "item": new_item
    }

In [7]:
df=pd.read_csv('items.csv')
len(df)

4

In [8]:
df['id'].max()

np.int64(4)

# Add this BELOW existing GET endpoint.

Test in Swagger: `http://127.0.0.1:8000/docs`

In [1]:
example = {
    "name": "Keyboard",
    "price": 1500,
    "quantity": 3
}
print(example)

{'name': 'Keyboard', 'price': 1500, 'quantity': 3}


# Step 8 — UPDATE Operation

Now we will update existing rows.

In [ ]:
from fastapi import HTTPException

# Add this BELOW: `from fastapi import FastAPI`

In [ ]:

# -----------------------------------
# UPDATE Operation
# Update existing item
# -----------------------------------
@app.put("/items/{item_id}")
def update_item(item_id: int, item: Item):

    # Read CSV
    df = pd.read_csv("items.csv")

    # Check if item exists
    if item_id not in df["id"].values:
        return {"error": "Item not found"}

    # Update matching row
    df.loc[df["id"] == item_id, "name"] = item.name
    df.loc[df["id"] == item_id, "price"] = item.price
    df.loc[df["id"] == item_id, "quantity"] = item.quantity

    # Save updated CSV
    df.to_csv("items.csv", index=False)

    return {
        "message": "Item updated successfully",
        "updated_item": {
            "id": item_id,
            "name": item.name,
            "price": item.price,
            "quantity": item.quantity
        }
    }


In [ ]:
example = {
    "name": "Keyboard",
    "price": 1500,
    "quantity": 3
}
print(example)

# Add this BELOW the POST endpoint.

Test using Swagger UI.

# Step 9 — DELETE Operation

Now we will remove rows from CSV.

In [ ]:

# -----------------------------------
# DELETE Operation
# Delete existing item
# -----------------------------------
@app.delete("/items/{item_id}")
def delete_item(item_id: int):

    # Read CSV
    df = pd.read_csv("items.csv")

    # Check if item exists
    if item_id not in df["id"].values:
        return {"error": "Item not found"}

    # Remove matching row
    df = df[df["id"] != item_id]

    # Save updated CSV
    df.to_csv("items.csv", index=False)

    return {
        "message": f"Item {item_id} deleted successfully"
    } 

In [3]:
import pandas as pd
df=pd.read_csv('items.csv')

In [7]:
df

,id,name,price,quantity
0,1,Laptop,50000.0,2
1,2,Phone,20000.0,5
2,3,Mouse,500.0,10
3,4,Keyboard,1500.0,7
4,5,Samsung phone,1000.0,12
5,6,Samsung phone,1000.0,12
6,7,Samsung phone,1000.0,12
7,8,Iphone,1000.0,12
8,9,ps5,60000.0,2


In [8]:
item_id=5

In [11]:
df

,id,name,price,quantity
0,1,Laptop,50000.0,2
1,2,Phone,20000.0,5
2,3,Mouse,500.0,10
3,4,Keyboard,1500.0,7
4,5,Samsung phone,1000.0,12
5,6,Samsung phone,1000.0,12
6,7,Samsung phone,1000.0,12
7,8,Iphone,1000.0,12
8,9,ps5,60000.0,2


In [9]:
df[df["id"] != item_id]

,id,name,price,quantity
0,1,Laptop,50000.0,2
1,2,Phone,20000.0,5
2,3,Mouse,500.0,10
3,4,Keyboard,1500.0,7
5,6,Samsung phone,1000.0,12
6,7,Samsung phone,1000.0,12
7,8,Iphone,1000.0,12
8,9,ps5,60000.0,2


# Add this BELOW the UPDATE endpoint.

Test DELETE using Swagger UI.

# Final Backend Completed

You now have:
- GET API
- POST API
- UPDATE API
- DELETE API
- CSV persistence
- Validation
- Error handling

In [ ]:
print("Swagger UI:")
print("http://127.0.0.1:8000/docs")